In [1]:
!nvidia-smi


Sun Mar 29 04:35:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/saccharomycetes/mllms_know.git
%cd /content/mllms_know


fatal: destination path 'mllms_know' already exists and is not an empty directory.
/content/mllms_know


In [3]:
!pip install -r requirements.txt


In [4]:
%cd /content/mllms_know/transformers
!pip install -e .
%cd /content/mllms_know


/content/mllms_know/transformers
Obtaining file:///content/mllms_know/transformers
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 105.9 MB/s eta 0:00:00
  Building editable for transformers (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-4.47.0.dev0-0.editable-py3-none-any.whl size=17428 sha256=03e76efa369eac8f4e0db2b968fdebd78dd72c871a5bdf0af978a4f4b636e00c
  Stored in directory: /tmp/pip-ephem-wheel-cache-za355uon/wheels/1c/22/58/56e810460716a81d9d431990217451c4330ac0196c9d94c6b4
Successfully built transformers
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfu

In [5]:
!mkdir -p /content/mllms_know/data/textvqa/images
!wget https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip -P /content/mllms_know/data/textvqa/images
!unzip -q /content/mllms_know/data/textvqa/images/train_val_images.zip -d /content/mllms_know/data/textvqa/images
!rm /content/mllms_know/data/textvqa/images/train_val_images.zip
!mv /content/mllms_know/data/textvqa/images/train_images/* /content/mllms_know/data/textvqa/images/
!rm -r /content/mllms_know/data/textvqa/images/train_images
!wget https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json -P /content/mllms_know/data/textvqa


--2026-03-29 04:37:05--  https://dl.fbaipublicfiles.com/textvqa/images/train_val_images.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 18.239.50.18, 18.239.50.104, 18.239.50.9, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|18.239.50.18|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7072297970 (6.6G) [application/zip]
Saving to: ‘/content/mllms_know/data/textvqa/images/train_val_images.zip’

train_val_images.zi 100%[===================>]   6.59G  27.2MB/s    in 4m 16s  

2026-03-29 04:41:22 (26.4 MB/s) - ‘/content/mllms_know/data/textvqa/images/train_val_images.zip’ saved [7072297970/7072297970]

--2026-03-29 04:42:47--  https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 18.239.50.9, 18.239.50.18, 18.239.50.104, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|18.239.50.9|:443... connected.
HTTP request sent, awaiting response... 

In [6]:
import json

with open('/content/mllms_know/data/textvqa/TextVQA_0.5.1_val.json') as f:
    datas = json.load(f)

new_datas = []
for data_id, data in enumerate(datas['data']):
    data_id = str(data_id).zfill(10)
    question = data['question']
    labels = data['answers']
    image_path = f"{data['image_id']}.jpg"
    new_data = {
        'id': data_id,
        'question': question,
        'labels': labels,
        'image_path': image_path
    }
    new_datas.append(new_data)

with open('/content/mllms_know/data/textvqa/data.json', 'w') as f:
    json.dump(new_datas, f, indent=4)

print("Total examples:", len(new_datas))


Total examples: 5000


In [7]:
import json
import shutil

full_path = "/content/mllms_know/data/textvqa/data.json"
backup_path = "/content/mllms_know/data/textvqa/data_full_backup.json"

shutil.copy(full_path, backup_path)

with open(full_path) as f:
    full_data = json.load(f)

small_data = full_data[:1]

with open(full_path, "w") as f:
    json.dump(small_data, f, indent=4)

print("Smoke-test examples:", len(small_data))
print(small_data[0])


Smoke-test examples: 1
{'id': '0000000000', 'question': 'what is the brand of this camera?', 'labels': ['nous les gosses', 'dakota', 'clos culombu', 'dakota digital', 'dakota', 'dakota', 'dakota digital', 'dakota digital', 'dakota', 'dakota'], 'image_path': '003a8ae2ef43b901.jpg'}


In [8]:
from pathlib import Path
import re

# Fix run.py by replacing everything from the top through the argparse import block
run_path = Path("/content/mllms_know/run.py")
run_text = run_path.read_text()

run_text = re.sub(
    r"^import os.*?import argparse\n",
    """import os
from PIL import Image
import torch
import numpy as np
from transformers import AutoProcessor, LlavaForConditionalGeneration, InstructBlipProcessor, InstructBlipForConditionalGeneration

try:
    from transformers import Qwen2_5_VLForConditionalGeneration
except ImportError:
    Qwen2_5_VLForConditionalGeneration = None

import argparse
""",
    run_text,
    flags=re.S,
)

run_path.write_text(run_text)

# Fix utils.py by replacing everything from the top through the base64 import block
utils_path = Path("/content/mllms_know/utils.py")
utils_text = utils_path.read_text()

utils_text = re.sub(
    r"^import torchvision\.transforms\.functional as TF.*?import base64\n",
    """import torchvision.transforms.functional as TF
import numpy as np
import os
from scipy.ndimage import median_filter
from skimage.measure import block_reduce

try:
    from qwen_vl_utils import process_vision_info
except ImportError:
    process_vision_info = None

from io import BytesIO
import base64
""",
    utils_text,
    flags=re.S,
)

utils_path.write_text(utils_text)

print("Cleanly rewrote import sections in run.py and utils.py")


Cleanly rewrote import sections in run.py and utils.py


In [9]:
with open("/content/mllms_know/utils.py") as f:
    for i in range(15):
        print(f.readline(), end="")


import torchvision.transforms.functional as TF
import numpy as np
import os
from scipy.ndimage import median_filter
from skimage.measure import block_reduce

try:
    from qwen_vl_utils import process_vision_info
except ImportError:
    process_vision_info = None

from io import BytesIO
import base64

def encode_base64(image):


In [14]:
!pip install bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 76.2 MB/s eta 0:00:00


In [15]:
from pathlib import Path

path = Path("/content/mllms_know/run.py")
text = path.read_text()

old = """    if args.model == 'llava':
        model = LlavaForConditionalGeneration.from_pretrained(args.model_id, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True, attn_implementation="eager").to(args.device)
        processor = AutoProcessor.from_pretrained(args.model_id)"""

new = """    if args.model == 'llava':
        from transformers import BitsAndBytesConfig
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        model = LlavaForConditionalGeneration.from_pretrained(
            args.model_id,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            attn_implementation="eager",
            quantization_config=quantization_config,
            device_map="auto",
        )
        processor = AutoProcessor.from_pretrained(args.model_id)"""

text = text.replace(old, new)
path.write_text(text)

print("Patched run.py for 4-bit LLaVA loading")


Patched run.py for 4-bit LLaVA loading


In [16]:
!mkdir -p /content/mllms_know/data/results
!python run.py --task textvqa --model llava --method rel_att --save_path /content/mllms_know/data/results


2026-03-29 04:56:46.084803: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774760206.346115    8268 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774760206.412645    8268 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774760206.927069    8268 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774760206.927115    8268 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774760206.927123    8268 computation_placer.cc:177] computation placer alr

In [17]:
!ls -lh /content/mllms_know/data/results


total 4.0K
-rw-r--r-- 1 root root 645 Mar 29 04:58 llava-textvqa-rel_att.json


In [18]:
import json
from pprint import pprint

result_path = "/content/mllms_know/data/results/llava-textvqa-rel_att.json"

with open(result_path) as f:
    results = json.load(f)

print("Num results:", len(results))
pprint(results[0])


Num results: 1
{'bbox': [33.0, 103.0, 435.0, 505.0],
 'crop_answer': 'Dakota digital',
 'id': '0000000000',
 'image_path': './data/textvqa/images/003a8ae2ef43b901.jpg',
 'labels': ['nous les gosses',
            'dakota',
            'clos culombu',
            'dakota digital',
            'dakota',
            'dakota',
            'dakota digital',
            'dakota digital',
            'dakota',
            'dakota'],
 'original_answer': 'Dakota digital',
 'question': 'what is the brand of this camera?'}


In [19]:
!python get_score.py --data_dir /content/mllms_know/data/results --save_path /content/mllms_know


/content/mllms_know/get_score.py:34: SyntaxWarning: invalid escape sequence '\d'
  periodStrip  = re.compile("(?!<=\d)(\.)(?!\d)")
/content/mllms_know/get_score.py:35: SyntaxWarning: invalid escape sequence '\d'
  commaStrip   = re.compile("(\d)(\,)(\d)")
100% 1/1 [00:00<00:00, 753.42it/s]
  model_name     task   method  raw_acc  crop_acc
0      llava  textvqa  rel_att     90.0      90.0


In [20]:
import json

with open("/content/mllms_know/evaluation_report.json") as f:
    report = json.load(f)

report


[{'model_name': 'llava',
  'task': 'textvqa',
  'method': 'rel_att',
  'raw_acc': 89.99999999999999,
  'crop_acc': 89.99999999999999}]

In [21]:
import shutil
shutil.copy(
    "/content/mllms_know/data/textvqa/data_full_backup.json",
    "/content/mllms_know/data/textvqa/data.json"
)
print("Full dataset restored")


Full dataset restored


In [22]:
import json

full_path = "/content/mllms_know/data/textvqa/data.json"

with open(full_path) as f:
    full_data = json.load(f)

small_data = full_data[:3]

with open(full_path, "w") as f:
    json.dump(small_data, f, indent=4)

print("Now testing with", len(small_data), "examples")


Now testing with 3 examples


In [23]:
!rm -f /content/mllms_know/data/results/llava-textvqa-rel_att.json


In [24]:
!python run.py --task textvqa --model llava --method rel_att --save_path /content/mllms_know/data/results


2026-03-29 05:04:25.099574: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774760665.130570   10340 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774760665.141003   10340 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774760665.176693   10340 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774760665.176735   10340 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774760665.176740   10340 computation_placer.cc:177] computation placer alr

In [25]:
!python get_score.py --data_dir /content/mllms_know/data/results --save_path /content/mllms_know


/content/mllms_know/get_score.py:34: SyntaxWarning: invalid escape sequence '\d'
  periodStrip  = re.compile("(?!<=\d)(\.)(?!\d)")
/content/mllms_know/get_score.py:35: SyntaxWarning: invalid escape sequence '\d'
  commaStrip   = re.compile("(\d)(\,)(\d)")
100% 3/3 [00:00<00:00, 1624.44it/s]
  model_name     task   method  raw_acc   crop_acc
0      llava  textvqa  rel_att     30.0  63.333333


In [26]:
import shutil
shutil.copy(
    "/content/mllms_know/data/textvqa/data_full_backup.json",
    "/content/mllms_know/data/textvqa/data.json"
)
print("Full dataset restored")


Full dataset restored


In [27]:
import json

full_path = "/content/mllms_know/data/textvqa/data.json"

with open(full_path) as f:
    full_data = json.load(f)

small_data = full_data[:5]

with open(full_path, "w") as f:
    json.dump(small_data, f, indent=4)

print("Now testing with", len(small_data), "examples")


Now testing with 5 examples


In [28]:
!rm -f /content/mllms_know/data/results/llava-textvqa-rel_att.json


In [29]:
!python run.py --task textvqa --model llava --method rel_att --save_path /content/mllms_know/data/results


2026-03-29 05:10:00.547288: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774761000.577367   11842 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774761000.586875   11842 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774761000.623300   11842 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774761000.623350   11842 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774761000.623356   11842 computation_placer.cc:177] computation placer alr

In [30]:
!python get_score.py --data_dir /content/mllms_know/data/results --save_path /content/mllms_know


/content/mllms_know/get_score.py:34: SyntaxWarning: invalid escape sequence '\d'
  periodStrip  = re.compile("(?!<=\d)(\.)(?!\d)")
/content/mllms_know/get_score.py:35: SyntaxWarning: invalid escape sequence '\d'
  commaStrip   = re.compile("(\d)(\,)(\d)")
100% 5/5 [00:00<00:00, 920.25it/s]
  model_name     task   method  raw_acc  crop_acc
0      llava  textvqa  rel_att     58.0      78.0


In [31]:
import shutil
shutil.copy(
    "/content/mllms_know/data/textvqa/data_full_backup.json",
    "/content/mllms_know/data/textvqa/data.json"
)
print("Full dataset restored")


Full dataset restored


In [32]:
import json

full_path = "/content/mllms_know/data/textvqa/data.json"

with open(full_path) as f:
    full_data = json.load(f)

small_data = full_data[:10]

with open(full_path, "w") as f:
    json.dump(small_data, f, indent=4)

print("Now testing with", len(small_data), "examples")


Now testing with 10 examples


In [33]:
!rm -f /content/mllms_know/data/results/llava-textvqa-rel_att.json


In [34]:
!python run.py --task textvqa --model llava --method rel_att --save_path /content/mllms_know/data/results


2026-03-29 05:14:37.213297: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774761277.243097   13085 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774761277.252453   13085 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774761277.286983   13085 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774761277.287030   13085 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774761277.287037   13085 computation_placer.cc:177] computation placer alr

In [35]:
!python get_score.py --data_dir /content/mllms_know/data/results --save_path /content/mllms_know


/content/mllms_know/get_score.py:34: SyntaxWarning: invalid escape sequence '\d'
  periodStrip  = re.compile("(?!<=\d)(\.)(?!\d)")
/content/mllms_know/get_score.py:35: SyntaxWarning: invalid escape sequence '\d'
  commaStrip   = re.compile("(\d)(\,)(\d)")
100% 10/10 [00:00<00:00, 1661.77it/s]
  model_name     task   method  raw_acc  crop_acc
0      llava  textvqa  rel_att     39.0      69.0


In [36]:
import shutil
shutil.copy(
    "/content/mllms_know/data/textvqa/data_full_backup.json",
    "/content/mllms_know/data/textvqa/data.json"
)
print("Full dataset restored")


Full dataset restored


In [37]:
!rm -f /content/mllms_know/data/results/llava-textvqa-rel_att.json


In [38]:
!python run.py --task textvqa --model llava --method rel_att --save_path /content/mllms_know/data/results --total_chunks 20 --chunk_id 0


2026-03-29 05:19:52.387178: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774761592.417988   14495 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774761592.426365   14495 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774761592.460161   14495 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774761592.460199   14495 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774761592.460206   14495 computation_placer.cc:177] computation placer alr

In [39]:
!python get_score.py --data_dir /content/mllms_know/data/results --save_path /content/mllms_know


/content/mllms_know/get_score.py:34: SyntaxWarning: invalid escape sequence '\d'
  periodStrip  = re.compile("(?!<=\d)(\.)(?!\d)")
/content/mllms_know/get_score.py:35: SyntaxWarning: invalid escape sequence '\d'
  commaStrip   = re.compile("(\d)(\,)(\d)")
100% 250/250 [00:00<00:00, 1757.55it/s]
  model_name     task   method  raw_acc  crop_acc
0      llava  textvqa  rel_att    42.12     53.16
